# 4 · Push: the robot events, live

This time the negotiation takes two calls: the `bff-rpc` routes run every intermediate step on their own. Then the consumer opens a push transfer: the provider subscribes to the factory's robot events and the consumer delivers each one to an endpoint of yours, here a small HTTP server inside this notebook.

In [ ]:
%run ./02_catalog.ipynb

In [ ]:
steps = []   # every DSP step of this notebook, for the timeline at the end


def rpc(base, path, payload, title=None):
    """POST one DSP step to an agent, show the call and keep it for the timeline."""
    response = requests.post(base + path, headers=json_header, data=json.dumps(payload))
    body = show(response, title)
    steps.append({
        "agent": helpers.AGENTS.get(base, base),
        "step": path.split("/dsp/current/", 1)[-1],
        "state": str(helpers.state_of(body) or "").rsplit(":", 1)[-1],
        "ms": int(response.elapsed.total_seconds() * 1000),
    })
    return body


def timeline(title):
    table(steps, [("Agent", "agent"), ("Step", "step"), ("State", "state"), ("ms", "ms")], title=title)


# The offer goes back exactly as the catalog advertises it.
offer = {
    "@id": policy_id,
    "@type": "Offer",
    "target": target_id,
    "permission": policy.get("permission") or [],
    "obligation": policy.get("obligation") or [],
    "prohibition": policy.get("prohibition") or [],
}

# Contract negotiation, the short way

## The consumer requests the offer (consumer → provider)

In [ ]:
payload = {
    "associatedAgentPeer": provider_did,
    "providerAddress": provider_dsp,
    "callbackAddress": consumer_dsp,
    "offer": offer,
}
body = rpc(consumer, "/dsp/current/negotiations/bff-rpc/setup-request-init", payload, "Contract request")
pids = {"consumerPid": body["response"]["consumerPid"], "providerPid": body["response"]["providerPid"]}

## The provider agrees, and the negotiation runs to the end (provider → consumer)

The answer carries the whole negotiation: every message the two agents exchanged, and the agreement.

In [ ]:
body = rpc(provider, "/dsp/current/negotiations/bff-rpc/setup-agreement", pids, "Agreement, and the rest on its own", )
negotiation = body["negotiationAgentModel"]
agreement = negotiation["agreement"]
agreement_id = agreement["id"]

# The provider's record: INBOUND messages came from the consumer, OUTBOUND ones it sent.
def sender(message):
    pair = (consumer_name, provider_name) if message.get("direction") == "INBOUND" else (provider_name, consumer_name)
    return " → ".join(pair)


table(
    negotiation.get("messages") or [],
    [
        ("Message", "messageType"),
        ("Sent", sender),
        ("State", lambda m: f"{m.get('stateTransitionFrom', '')} → {m.get('stateTransitionTo', '')}"),
        ("At", "createdAt"),
    ],
    title=f"What the two agents exchanged · agreement {agreement_id}",
);

# Push transfer

## An endpoint for the events

The consumer's agent POSTs every event here. It runs in its container, so it reaches this notebook at `host.docker.internal`.

In [ ]:
import queue
import threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

events = queue.Queue()


class Sink(BaseHTTPRequestHandler):
    def do_POST(self):
        raw = self.rfile.read(int(self.headers.get("Content-Length") or 0))
        try:
            events.put(json.loads(raw))
        except ValueError:
            events.put(raw.decode("utf-8", errors="replace"))
        self.send_response(200)
        self.end_headers()

    def log_message(self, *args):
        pass


sink_port = int(env("EUNOMIA_SINK_PORT"))
sink_url = env("EUNOMIA_SINK_DOCKER_URL")
sink = ThreadingHTTPServer(("0.0.0.0", sink_port), Sink)
threading.Thread(target=sink.serve_forever, daemon=True).start()
print(f"Listening on :{sink_port}; the consumer pushes to {sink_url}")

## The consumer requests a push transfer, with the endpoint (consumer → provider)

In [ ]:
payload = {
    "associatedAgentPeer": provider_did,
    "providerAddress": provider_dsp,
    "callbackAddress": consumer_dsp,
    "agreementId": agreement_id,
    "format": "HTTP_PUSH",
    "dataAddress": {"endpointType": "HTTP", "endpoint": sink_url, "endpointProperties": []},
}
body = rpc(consumer, "/dsp/current/transfers/rpc/setup-request", payload, "Push transfer request")
transfer = {"consumerPid": body["response"]["consumerPid"], "providerPid": body["response"]["providerPid"]}

## The provider starts it, and subscribes to the factory (provider → consumer)

In [ ]:
rpc(provider, "/dsp/current/transfers/rpc/setup-start", transfer, "Transfer start: the provider subscribes to the factory");

## The events arrive

The factory emits one every few seconds. The table below fills in as they arrive: five events, or 60 seconds.

In [ ]:
import time

EVENT_COLUMNS = [
    ("#", "n"),
    ("Arrived", "arrived"),
    ("Event", lambda e: e.get("item_type", "")),
    ("Value", lambda e: e.get("last_value")),
    ("Measured at", lambda e: e.get("last_measured_at", "")),
]

received = []
started = time.monotonic()
live = table(received, EVENT_COLUMNS, title="Robot events, as they arrive")
while len(received) < 5 and time.monotonic() - started < 60:
    try:
        event = events.get(timeout=max(0.1, 60 - (time.monotonic() - started)))
    except queue.Empty:
        break
    if not isinstance(event, dict):
        event = {"item_type": str(event)}
    received.append({**event, "n": len(received) + 1, "arrived": f"+{time.monotonic() - started:.1f} s"})
    table(received, EVENT_COLUMNS, title=f"Robot events, as they arrive · {len(received)} so far", handle=live)

if not received:
    raise RuntimeError(f"No event arrived. Look at: ./eunomia logs {provider_name}, ./eunomia logs {consumer_name}, ./eunomia logs demo")

One event, as the consumer delivered it:

In [ ]:
print(json.dumps({k: v for k, v in received[0].items() if k not in ("n", "arrived")}, indent=2))

## The provider completes the transfer, and unsubscribes (provider → consumer)

In [ ]:
rpc(provider, "/dsp/current/transfers/rpc/setup-completion", transfer, "Completion: the provider unsubscribes")
sink.shutdown()
sink.server_close()
timeline("Everything this notebook asked of the agents");